# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access Croissant metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their `@id` keys.

Here we list all record sets, their fields, and columns as defined by the Croissant schema. Each entity is referenced by its `@id`, ensuring precise reference for further exploration.

In [ ]:
# List record sets and fields using @id

# Get all record sets in the dataset
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets defined in the top-level metadata. Searching for record sets in 'hasPart' if present...")
    # Some datasets use hasPart to link record sets
    if hasattr(metadata, 'hasPart'):
        has_parts = metadata.hasPart if isinstance(metadata.hasPart, list) else [metadata.hasPart]
        record_sets = [p for p in has_parts if getattr(p, 'type', None) == 'RecordSet' or getattr(p, '@type', None) == 'RecordSet']
else:
    record_sets = record_sets if isinstance(record_sets, list) else [record_sets]

if not record_sets:
    # As mlcroissant >=0.7.0: try with dataset.record_sets()
    record_sets = list(dataset.record_sets())

record_sets_ids = []

for rs in record_sets:
    # rs may be a dict, mlcroissant object, or just an ID
    # Prefer objects with .fields, .columns etc.
    try:
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None) or rs
        print(f"Record Set @id: {rs_id}")
        record_sets_ids.append(rs_id)
        # Fields
        if hasattr(rs, 'field'):
            fields = rs.field
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                field_id = getattr(field, '@id', None) or getattr(field, 'id', None) or field
                print(f"\tField @id: {field_id}")
                # Columns for each field
                if hasattr(field, 'column'):
                    columns = field.column
                    if not isinstance(columns, list):
                        columns = [columns]
                    for col in columns:
                        col_id = getattr(col, '@id', None) or getattr(col, 'id', None) or col
                        print(f"\t\tColumn @id: {col_id}")
    except Exception as e:
        print(f"Could not parse record set structure: {e}")

# Alternatively, print IDs found directly
if not record_sets_ids and hasattr(metadata, 'distribution'):
    print("\nNo explicit record sets, will infer from distributions (files):\n")
    for dist in metadata.distribution:
        dist_id = getattr(dist, "@id", None) or dist
        print(f"Distribution @id: {dist_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, we use the primary available record set or infer from the first provided record set or distribution.

In [ ]:
# Identify the main record set to load (by @id)

# If we discovered record set IDs above
if record_sets_ids:
    record_sets_to_use = record_sets_ids
else:
    # If not, fall back to distribution @ids
    record_sets_to_use = []
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            dist_id = getattr(dist, "@id", None) or dist
            record_sets_to_use.append(dist_id)

dataframes = {}

for record_set_id in record_sets_to_use:
    try:
        # Attempt to load records by record_set @id
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set '@id': {record_set_id}")
            print("Columns:", df.columns.tolist())
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records from {record_set_id} due to error: {e}")

# Display a quick sample
if dataframes:
    # Pick the first DataFrame for demonstration
    first_id = next(iter(dataframes.keys()))
    print(f"\nFirst rows from record set '{first_id}':")
    display_columns = dataframes[first_id].columns.tolist()
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded. Please check record set IDs or dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping by attributes. All field access should use the proper `@id`/column name as previously displayed.

You may need to adjust the field names below according to the actual field `@id`s in your data.

In [ ]:
# Pick the first available DataFrame for demonstration
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    # List numeric columns to help user select
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    print("Numeric columns:", numeric_cols)
    if numeric_cols:
        # Use the first numeric column found for demonstration
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()  # Example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {filtered_df.shape[0]} records")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by a categorical field (if any)
        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        if cat_cols:
            group_field = cat_cols[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No numeric fields found to analyze.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust field names as needed to match the dataset columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot histogram of first numeric column
if dataframes:
    df = dataframes[next(iter(dataframes.keys()))]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        field_to_plot = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[field_to_plot].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {field_to_plot}")
        plt.xlabel(field_to_plot)
        plt.ylabel("Frequency")
        plt.show()
        
        # If categorical columns exist, boxplot
        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        if cat_cols:
            group_field = cat_cols[0]
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=df, x=group_field, y=field_to_plot)
            plt.title(f"{field_to_plot} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key insights and next steps.

- This notebook demonstrated loading and analyzing a dataset using the `mlcroissant` library.
- We explored record sets, fields, and performed filtering, normalization, grouping, and visualization.
- For richer analysis and machine learning, further domain-specific exploration and feature selection is recommended.

Refer to the [FAIR^2 dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for metadata details and curation workflow.